# 🎙️ Garuita Cuentos: probar voz de narrador clonada (Chatterbox)

**Cómo usarlo (no hace falta saber programar):**

1. Arriba, en el menú: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar**.
2. Apretá el botón ▶️ de cada paso, **en orden**, y esperá a que termine (aparece un ✅).
3. En el **Paso 3** subís tu grabación (10 a 15 segundos, vos hablando claro, sin música ni ruido).
   Si no subís nada, usa una voz de ejemplo que trae el programa.
4. En el **Paso 4** escribís el texto y te devuelve el audio para escuchar y descargar.
5. En el **Paso 5** subís un cuento (.json) y te devuelve un .zip con la narración de cada escena, listo para el motor.

⚠️ Solo clonar voces propias o con permiso escrito del dueño.

ℹ️ Si en el Paso 1 aparece un aviso de **"Reiniciar sesión"**, aceptalo y seguí con el Paso 2.

In [ ]:
#@title Paso 1: Instalar (tarda unos 3 a 5 minutos, solo la primera vez)
import subprocess, sys
try:
    gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
if not gpu:
    print("⚠️ No hay GPU. Andá a Entorno de ejecución → Cambiar tipo de entorno → T4 GPU, y volvé a correr este paso.")
else:
    print("GPU encontrada:", gpu)
    !pip install -q chatterbox-tts==0.1.7
    # Colab trae torchvision para otra versión de torch y rompe la carga; no lo usamos
    !pip uninstall -y -q torchvision
    print("✅ Listo. Si aparece un aviso para reiniciar la sesión, aceptalo y seguí con el Paso 2.")

In [ ]:
#@title Paso 2: Cargar el modelo de voz (1 a 2 minutos)
import torch
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
modelo = ChatterboxMultilingualTTS.from_pretrained(device=dispositivo)
print("✅ Modelo cargado en", dispositivo)

In [ ]:
#@title Paso 3: Subir tu grabación de voz (opcional)
#@markdown Apretá ▶️ y después **Elegir archivos**. Sirve mp3, wav, m4a, ogg u opus (como los audios de WhatsApp).
#@markdown Si no querés subir nada, apretá **Cancelar subida** y se usa la voz de ejemplo.
import os, subprocess
from google.colab import files
voz_referencia = None
try:
    subidos = files.upload()
except Exception:
    subidos = {}
if subidos:
    original = list(subidos.keys())[0]
    voz_referencia = "/content/mi_voz.wav"
    # Pasa a WAV mono y se queda con los primeros 15 segundos
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", original,
                    "-ac", "1", "-ar", "24000", "-t", "15", voz_referencia], check=True)
    print("✅ Voz lista:", original)
else:
    print("ℹ️ No subiste voz: se usará la voz de ejemplo del programa.")

In [ ]:
#@title Paso 4: Escribir el texto y generar la narración
texto = "Había una vez, junto a un río de agua clara, una pequeña rana llamada Garuita. Cada mañana saludaba a los peces y escuchaba el canto de los pájaros. Pero un día, el río amaneció triste." #@param {type:"string"}
#@markdown **Expresividad**: más alto = más emoción y teatro (0.5 normal, 0.7 cuento animado).
expresividad = 0.85 #@param {type:"slider", min:0.25, max:1.5, step:0.05}
#@markdown **Ritmo**: más bajo = habla más pausado (0.3 a 0.5 suele sonar bien para cuentos).
ritmo = 0.5 #@param {type:"slider", min:0.0, max:1.0, step:0.05}

import re, numpy as np, soundfile as sf, subprocess
from IPython.display import Audio, display
from google.colab import files

# Corta el texto en frases de hasta ~250 letras (el modelo funciona mejor con trozos cortos)
frases = [f.strip() for f in re.split(r'(?<=[.!?…])\s+', texto.strip()) if f.strip()]
trozos, actual = [], ""
for f in frases:
    if actual and len(actual) + len(f) > 250:
        trozos.append(actual); actual = f
    else:
        actual = (actual + " " + f).strip()
if actual:
    trozos.append(actual)

silencio = np.zeros(int(modelo.sr * 0.35), dtype=np.float32)  # pausa corta entre frases
partes = []
for i, t in enumerate(trozos, 1):
    print(f"Generando parte {i} de {len(trozos)}...")
    wav = modelo.generate(t, language_id="es", audio_prompt_path=voz_referencia,
                          exaggeration=expresividad, cfg_weight=ritmo)
    partes += [wav.squeeze(0).numpy().astype(np.float32), silencio]

audio = np.concatenate(partes)
sf.write("/content/narracion.wav", audio, modelo.sr)
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "/content/narracion.wav",
                "-b:a", "192k", "/content/narracion.mp3"], check=True)
print(f"✅ Listo: {len(audio)/modelo.sr:.1f} segundos")
display(Audio("/content/narracion.wav"))
files.download("/content/narracion.mp3")

In [ ]:
#@title Paso 5: Narrar un cuento completo (para el motor)
#@markdown Usa la voz del Paso 3 y los ajustes del Paso 4.
#@markdown Apretá ▶️, subí el archivo del cuento (por ejemplo **rio-limpio.json**) y esperá.
#@markdown Al final se descarga **NOMBRE-voz.zip**: subilo al chat del proyecto.
import json, os, re, zipfile, numpy as np, soundfile as sf
from google.colab import files

def narrar(texto):
    frases = [f.strip() for f in re.split(r'(?<=[.!?…])\s+', texto.strip()) if f.strip()]
    trozos, actual = [], ""
    for f in frases:
        if actual and len(actual) + len(f) > 250:
            trozos.append(actual); actual = f
        else:
            actual = (actual + " " + f).strip()
    if actual:
        trozos.append(actual)
    pausa = np.zeros(int(modelo.sr * 0.35), dtype=np.float32)
    partes = []
    for i, t in enumerate(trozos):
        wav = modelo.generate(t, language_id="es", audio_prompt_path=voz_referencia,
                              exaggeration=expresividad, cfg_weight=ritmo)
        if i:
            partes.append(pausa)
        partes.append(wav.squeeze(0).numpy().astype(np.float32))
    return np.concatenate(partes)

subido = files.upload()
nombre_archivo = list(subido.keys())[0]
cuento = json.loads(subido[nombre_archivo].decode("utf-8"))
nombre = re.sub(r'[^a-zA-Z0-9_-]', '', os.path.splitext(nombre_archivo)[0].split(' ')[0]) or "cuento"
# Los mismos textos que muestra el motor (título hasta 40 letras, escenas hasta 140)
lineas = [("titulo", str(cuento.get("titulo", ""))[:40])]
for i, e in enumerate(cuento.get("escenas", []), 1):
    lineas.append((f"escena-{i:02d}", str(e.get("texto", ""))[:140]))

carpeta = f"/content/{nombre}-voz"
os.makedirs(carpeta, exist_ok=True)
for n, (archivo, texto) in enumerate(lineas, 1):
    if not texto.strip():
        continue
    print(f"[{n}/{len(lineas)}] {archivo}: {texto}")
    sf.write(f"{carpeta}/{archivo}.wav", narrar(texto), modelo.sr)

zip_final = f"/content/{nombre}-voz.zip"
with zipfile.ZipFile(zip_final, "w") as z:
    for f in sorted(os.listdir(carpeta)):
        z.write(f"{carpeta}/{f}", f)
print(f"✅ Listo: {zip_final}. Subilo al chat del proyecto.")
files.download(zip_final)